# BotChain AI — Single-Agent Prototype

Single `deepagents` agent (Plan + Build merged) wired to:
- **n8n-mcp** (via `npx`, stdio transport) for grounded node lookup + `validate_workflow`
- **Ollama Cloud** model via `langchain-ollama`
- **Sandbox filesystem backend** → writes generated workflow JSON to `project/sandbox/`
- **SQLite checkpointing** → chat/session persistence across turns and kernel restarts
- **LangSmith tracing** → full run visibility

Notebook lives in `project/notebook/`; sandbox lives in `project/sandbox/` (sibling dir).


## 1. Imports & environment

In [15]:
import os
import asyncio
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()  # expects a .env in project root (or notebook dir) — see next cell for required keys


True

**Required env vars** (put these in a `.env` file — never hardcode keys in the notebook):

```
OLLAMA_API_KEY=<your-ollama-cloud-api-key>
LANGSMITH_API_KEY=<your-langsmith-api-key>
```


In [16]:
# --- LangSmith tracing ---
# Turns on full run tracing for every agent invocation below — visible in your LangSmith project.
os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING", "true")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT", "botchain-ai")
os.environ["OLLAMA_API_KEY"] = os.getenv("OLLAMA_API_KEY")
os.environ["N8N_API_URL"] = os.getenv("N8N_API_URL")
os.environ["N8N_API_KEY"] = os.getenv("N8N_API_KEY")

## 2. Sandbox directory setup

`project/notebook/` → `project/sandbox/` (sibling directory). All agent file writes are confined here.

In [17]:
NOTEBOOK_DIR = Path.cwd()              # assumes Jupyter was launched from project/notebook
PROJECT_ROOT = NOTEBOOK_DIR.parent     # project/
SANDBOX_DIR = PROJECT_ROOT / "sandbox"
CHECKPOINT_DB = PROJECT_ROOT / "checkpoints.sqlite"

SANDBOX_DIR.mkdir(parents=True, exist_ok=True)
print(f"Sandbox ready at: {SANDBOX_DIR.resolve()}")
print(f"Checkpoint DB at: {CHECKPOINT_DB.resolve()}")


Sandbox ready at: /Volumes/Mitul/Projects/botchain-ai/sandbox
Checkpoint DB at: /Volumes/Mitul/Projects/botchain-ai/checkpoints.sqlite


## 3. n8n-mcp tools (via `npx`)

Spawns `n8n-mcp` as a stdio subprocess and loads its tools (`search_nodes`, `get_node_essentials`,
`get_node_documentation`, `validate_workflow`, etc.) as LangChain-compatible tools.

> Swap `"args": ["-y", "n8n-mcp"]` for your fork's entry point if you want to test against it
> instead of the published package, e.g. `["/path/to/your-fork/dist/index.js"]` with `"command": "node"`.


In [18]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient({
    "n8n-mcp": {
      "transport": "stdio",
      "command": "npx",
      "args": ["n8n-mcp"],
      "env": {
        "MCP_MODE": "stdio",
        "LOG_LEVEL": "error",
        "DISABLE_CONSOLE_OUTPUT": "true",
        "N8N_API_URL": "https://your-n8n-instance.com",
        "N8N_API_KEY": os.getenv("N8N_API_KEY")
      }
    }
})

mcp_tools = await mcp_client.get_tools()

print(f"Loaded {len(mcp_tools)} tools from n8n-mcp:")
for t in mcp_tools:
    print(f"  - {t.name}")


Loaded 24 tools from n8n-mcp:
  - tools_documentation
  - search_nodes
  - get_node
  - validate_node
  - get_template
  - search_templates
  - validate_workflow
  - n8n_create_workflow
  - n8n_get_workflow
  - n8n_update_full_workflow
  - n8n_update_partial_workflow
  - n8n_delete_workflow
  - n8n_list_workflows
  - n8n_validate_workflow
  - n8n_autofix_workflow
  - n8n_test_workflow
  - n8n_executions
  - n8n_evaluations
  - n8n_health_check
  - n8n_workflow_versions
  - n8n_deploy_template
  - n8n_manage_datatable
  - n8n_manage_credentials
  - n8n_audit_instance


## 4. Model — Ollama Cloud

Points `ChatOllama` at Ollama's cloud endpoint with your API key instead of a local server.
Swap `model=` for whichever cloud-hosted tag you're testing (must support tool calling).


In [ ]:
from langchain_ollama import ChatOllama

OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY", "<insert-ollama-cloud-api-key>")

model = ChatOllama(
    model="qwen3-coder:480b-cloud",  # any tool-calling-capable Ollama Cloud model tag
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": f"Bearer {OLLAMA_API_KEY}"}},
    temperature=0.2,
)


## 5. Checkpointer — SQLite session persistence

Requires `langgraph-checkpoint-sqlite` (`uv add langgraph-checkpoint-sqlite` if not already installed).
Using the async variant since we stream the agent with `astream`.


In [ ]:
import aiosqlite
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

conn = await aiosqlite.connect(str(CHECKPOINT_DB))
checkpointer = AsyncSqliteSaver(conn)


## 6. Build the agent

Single deep agent handling both Plan and Build phases internally (see system prompt).
`FilesystemBackend(root_dir=SANDBOX_DIR, virtual_mode=True)` gives the agent's built-in
`write_file` tool real disk access — but sandboxed to `SANDBOX_DIR`, blocking `..` traversal.


In [ ]:
# Paste the full system prompt generated earlier here — kept as a placeholder to avoid
# re-spending tokens regenerating it in this notebook.
SYSTEM_PROMPT = """<insert-prompt-here>"""


In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(root_dir=str(SANDBOX_DIR), virtual_mode=True)

agent = create_deep_agent(
    model=model,
    tools=mcp_tools,
    system_prompt=SYSTEM_PROMPT,
    backend=backend,
    checkpointer=checkpointer,
)


## 7. Streaming chat helper

Streams token-by-token via `stream_mode="messages"`. `thread_id` is what ties a conversation
to its checkpointed state — reuse the same `thread_id` across calls to continue a session,
even after a kernel restart.


In [ ]:
from langchain_core.messages import HumanMessage

async def chat(user_input: str, thread_id: str = "session-1"):
    """Send one user turn to the agent and stream the response as it's generated."""
    config = {"configurable": {"thread_id": thread_id}}

    print(f"User: {user_input}\n")
    print("Agent: ", end="", flush=True)

    async for chunk, metadata in agent.astream(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
        stream_mode="messages",
    ):
        # Only print actual text tokens (skip empty tool-call-only chunks)
        if getattr(chunk, "content", None):
            print(chunk.content, end="", flush=True)

    print("\n" + "-" * 60)


## 8. Test run

First turn starts a new session under `thread_id="demo-session-1"`. Run the second cell
afterward (same `thread_id`) to confirm persistence — the agent should remember the first turn.


In [ ]:
THREAD_ID = "demo-session-1"

await chat(
    "I want to automate lead triage: whenever a new row is added to my Google Sheet, "
    "check if the lead's company size is over 50, and if so post a summary to our #sales Slack channel.",
    THREAD_ID,
)


In [ ]:
# Continuing the SAME thread_id — tests that checkpointed state persists the conversation
await chat("Yes, that's correct. Go ahead and build it.", THREAD_ID)


In [ ]:
# Inspect what landed in the sandbox
list(SANDBOX_DIR.glob("*.json"))


## 9. Resuming a session later (new kernel, same thread_id)

Because `AsyncSqliteSaver` persists to `checkpoints.sqlite` on disk, re-running cells 1–6
after a full kernel restart and then calling `chat(..., thread_id="demo-session-1")` again
will resume the exact same conversation state — no need to replay earlier turns.
